# 06 — Asking a real agent fleet a plain-English question

Every notebook so far ran everything inside this one kernel process. This
one is different: it works against **real, separate `scarlet-agents`
containers** (one head, two workers), waits for them to come online, then
asks a question in plain English - the head's LLM decides which
`scarlet_agentic_harness` skills to call (here: `sum`, twice, with
different transforms) and reports back a natural-language answer, having
actually dispatched to and coordinated with the two worker containers over
Redis.

Each worker's private data is its own CSV file, generated by this notebook
into a shared Docker volume - real separate containers, so each just gets
its own `CSV_PATH`; no in-process trickery needed to keep them from
stepping on each other's data.

## Two ways the fleet gets deployed

This notebook does not care which one you use:

- **Locally (default)** - `examples/notebooks/docker-compose.yml` declares
  the three agent containers under a `agents` profile, which a plain
  `docker compose up` skips. This notebook brings them in on demand.
- **Through Gustavo** - deploy `gustavo-app.yaml` instead. Gustavo owns
  the fleet's lifecycle, and sets `GUSTAVO_MANAGED_AGENTS=true` so this
  notebook waits for the agents rather than launching (or tearing down)
  anything itself.

Either way the notebook talks to the fleet the same way: over Redis for
liveness, over the head's MCP endpoint to ask the question.

## Parameters

Everything that names an agent lives here. All of it is overridable by
environment variable, which is what lets the same notebook run unmodified
under Gustavo - the app config injects these, and the cell below picks
them up.

An agent's identity on the bus is `f"{APP_ID}_{NODE_ADDRESS}"`
(`HarnessConfig.agent_id`), so `APP_ID` plus the node addresses below are
exactly what the liveness check looks for **and** what the compose file
interpolates into the containers. One source of truth, so the fleet that
gets launched and the fleet that gets waited for cannot drift apart.

In [ ]:
import os, time, json, subprocess, random

# ─── identity ────────────────────────────────────────────────────────────
APP_ID       = os.environ.get("APP_ID", "tutorial_stats")
HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1,worker2").split(",")
                if n.strip()]

# The bus the harness head registers on, and the hostname its MCP server is
# reachable at. Under compose that hostname is the service name; under
# Gustavo it is whatever the app config called it.
HEAD_BUS  = os.environ.get("HEAD_BUS", f"{APP_ID}_headagent")
HEAD_HOST = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

# ─── derived: who we expect on the bus ───────────────────────────────────
HEAD_AGENT_ID      = f"{APP_ID}_{HEAD_NODE}"
WORKER_AGENT_IDS   = {f"{APP_ID}_{n}" for n in WORKER_NODES}
EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

# ─── deployment mode ─────────────────────────────────────────────────────
# false (the default) means this notebook owns the fleet: it brings the
# compose "agents" profile up below, and tears it down at the end. true
# means Gustavo deployed the agents and owns them - this notebook then
# only ever waits and observes, and the teardown cell is a no-op.
GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"app id        : {APP_ID}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"head bus      : {HEAD_BUS}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

## Generate each worker's private data

Two CSV files, one per worker, written into the `csv-data` volume this
container shares with them (read-write here, read-only there). The workers
read them through `data_connectors.CsvConnector`, which expects a single
`value` column - see `harness/scarlet_agentic_harness/skills/local_data.py`.

Generated rather than shipped as repo files or baked into an image: the
data belongs to the tutorial, not to any container, and this keeps the
published images free of example payloads. The seed is fixed so the
variance the agent reports at the end is reproducible run to run.

Under Gustavo the workers get their data from wherever that deployment
points `CSV_PATH`, so this cell only writes files when the notebook owns
the fleet.

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR", "/data")

def write_worker_csv(path, values):
    with open(path, "w") as f:
        f.write("value\n")
        for v in values:
            f.write(f"{v}\n")

if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo-managed fleet - workers use their own deployment's CSV_PATH, "
          "nothing generated here")
else:
    rng = random.Random(20260928)   # fixed seed: reproducible variance
    os.makedirs(DATA_DIR, exist_ok=True)
    for i, node in enumerate(WORKER_NODES, start=1):
        # Each worker gets a different distribution, so the fleet-wide
        # variance is genuinely a property of the combined data rather
        # than of either worker alone.
        values = [round(rng.gauss(20 + 2 * i, 1.5), 2) for _ in range(12)]
        path = f"{DATA_DIR}/worker{i}.csv"
        write_worker_csv(path, values)
        print(f"{node}: wrote {len(values)} readings to {path}")

## Bring the fleet up

One `docker compose --profile agents up -d` against the same compose file
that started this notebook - bind-mounted in at `/workspace`, driving the
host's Docker daemon through the mounted socket. Because the compose
project name is pinned in that file, these containers join the network the
already-running Redis is on rather than starting a second, isolated stack.

`up -d` is idempotent, so re-running this cell is safe: anything already
running is left alone, anything missing is started. That is also what makes
a partially-dead fleet self-healing.

When `GUSTAVO_MANAGED_AGENTS=true` this cell launches nothing at all.

In [ ]:
# The compose SERVICE names of the fleet - distinct from the agent ids on
# the bus above (those are "{APP_ID}_{NODE_ADDRESS}"). Every compose command
# below names these explicitly, and that is not cosmetic:
#
# `docker compose up -d` with no services named operates on the WHOLE
# project - including the jupyter service this notebook is running inside.
# Compose would then notice that service is out of date and recreate it,
# killing this kernel mid-cell. Verified directly (this session): the
# container took SIGTERM and the notebook died. Naming the services scopes
# every operation to the fleet and leaves the rest of the stack alone.
AGENT_SERVICES = ["agent-head", "agent-worker1", "agent-worker2"]


def compose(*args):
    """Run a compose command scoped to the agent services only.

    Failures print what Compose actually said: check=True on its own would
    raise CalledProcessError with the diagnostic buried in .stderr, which
    Jupyter renders as a bare traceback giving no hint what was wrong.
    """
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result


if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    print("bringing up the agents profile ...")
    # --no-deps: bring up ONLY these three, never their depends_on.
    # Redis is already running (this notebook could not be talking to
    # you otherwise), and without this Compose will happily decide the
    # running Redis is out of date and recreate it - dropping the bus
    # out from under any agent already registered on it. The bring-up
    # should only ever add containers, never disturb the stack.
    compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

## Wait for the fleet to come online

`up -d` returns as soon as the containers exist, well before the harness
inside them has connected to Redis and registered - and under Gustavo this
notebook has no idea when the rollout will land. So this is a real wait,
not a fixed sleep: poll `GatherStatus()` on the head bus until every
expected agent is there.

**Liveness is judged on freshness, not presence.** `Messenger.ReportStatus`
writes its registration record with a plain `SET` and no TTL
(`scarlets/messaging/Messenger.py`), so a dead agent's record sits in Redis
indefinitely. Checking only whether an id is *present* would happily match
a previous run's corpses and sail on into a fleet that isn't there - every
subsequent dispatch would then hang with no explanation. Each record
carries a `ts` refreshed by the heartbeat thread every 30 s, so an agent
counts as live only if its `ts` is recent.

In [ ]:
from scarlets.messaging import Messenger

# 3x the 30s heartbeat interval: tolerates one missed beat without a false
# negative, while a fleet dead longer than this is correctly seen as gone.
FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    """Agent ids on the head bus whose heartbeat is recent enough to trust."""
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS}: "
                f"{sorted(missing)}\nCheck {where}."
            )
        time.sleep(2)

# A remote Gustavo rollout is slower than a local container start.
wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

## Ask a plain-English question

The head decides, on its own, to call `sum` twice (`transform=identity` for
Sigma(x), `transform=square` for Sigma(x^2)) across both workers' real CSV
data, then composes mean/variance/stddev itself before replying - no
distributed "variance" skill exists, by design (see `sum.py`'s own
docstring).

The notebook's own process never calls the LLM. It only speaks MCP to the
head, which holds the LLM credentials; the workers never see them at all.

In [ ]:
import asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text

answer = asyncio.run(ask_scarlet_agent(
    "What are the sum, mean, and variance of the readings held across all "
    "currently online workers?"
))
print(answer)

## Teardown

Stops and removes the three agent containers, leaving Redis, the Composer
UI, and this notebook running - so you can re-run from the top without
restarting the whole stack.

Only the default services are left standing; `docker compose down` from
`examples/notebooks/` on the host takes those down too when you're done.

Under Gustavo this is deliberately a no-op: those containers belong to that
deployment, and a tutorial notebook has no business removing them.

In [ ]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    # `rm -sf <services>`, never a bare `compose down`: down would take the
    # whole project with it - Redis, the Composer UI, and the jupyter
    # container this notebook is running inside. Stop and remove only the
    # three agent containers, leaving the stack you started with intact.
    compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")